In [ ]:
import time
import sqlite3
import mysql.connector
import requests
from bs4 import BeautifulSoup

# SQLite DB와 테이블 생성 (없으면 생성)
conn = sqlite3.connect('Crawlingpractice.db')
cursor = conn.cursor()

base_url = 'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber='

for page in range(1, 6):
    url = base_url + str(page)
    print(f'크롤링 중: {url}')
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    best_list_el = soup.select('#yesBestList div.item_info')
    
    for i, item in enumerate(best_list_el):
        title = item.select_one('div.info_name > a').text.strip()
        author = item.select_one('div.info_pubGrp a').text.strip()
        price = item.select_one('.info_price > .txt_num').text.strip()
        
        rank = (page - 1) * 24 + i + 1
        
        cursor.execute('''
            INSERT OR REPLACE INTO books (rank, title, author, price)
            VALUES (?, ?, ?, ?)
        ''', (rank, title, author, price))
    
    conn.commit()
    time.sleep(1)  # 서버 과부하 방지

conn.close()
print("크롤링 및 DB 저장 완료")

In [ ]:
import sqlite3

conn = sqlite3.connect('my_database.db')
sqlite_cursor = conn.cursor()

sqlite_cursor.execute("""
    SELECT RANK, title, author, CAST(REPLACE(price, ',', '') AS INTEGER) AS price_clean
    FROM books
""")

rows = sqlite_cursor.fetchall()
rows

In [ ]:
# MySQL 연결 (Workbench가 연결된 서버 정보 입력)
mysql_conn = mysql.connector.connect(
    host='localhost',         
    user='root',             
    password='1234', 
    database='bookstore'      
)
mysql_cursor = mysql_conn.cursor()

In [ ]:
insert_query = "REPLACE INTO books (book_rank, title, author, price) VALUES (%s, %s, %s, %s)"
for row in rows:
    mysql_cursor.execute(insert_query, row)


In [ ]:
# 커밋 및 연결 종료
mysql_conn.commit()
sqlite_conn.close()
mysql_conn.close()
